# Prerequisites

## Load Bronze data to dataframe

In [0]:
%sql
use catalog ecommerce;
create schema if not exists silver;


In [0]:
clickstream_df = spark.read.table("bronze.clickstream")
display(clickstream_df.count())
clickstream_df.limit(5).display()

In [0]:
clickstream_df.printSchema()

# Transformations

## Drop Duplicates

In [0]:
from pyspark.sql.functions import *
clickstream_df = clickstream_df.dropDuplicates()
display(clickstream_df.count())
clickstream_df.limit(5).display()

## Validate timestamps.

In [0]:
clickstream_df = clickstream_df.filter(col("event_time") < current_timestamp()).filter(
    col("event_time").isNotNull()
)
display(clickstream_df.count())
clickstream_df.limit(5).display()

## Standardize Standardize event names

In [0]:
clickstream_df.select(col("event_type")).distinct().display()

In [0]:
clickstream_df = clickstream_df.withColumn("event_type", upper(trim(col("event_type"))))

display(clickstream_df.count())
clickstream_df.limit(5).display()

## Validate product IDs , customer IDs where present

In [0]:
clickstream_df = clickstream_df.filter(
    (col("customer_id").isNull() | (col("customer_id") > 0))
).filter((col("product_id").isNull() | (col("product_id") > 0)))
display(clickstream_df.count())
clickstream_df.limit(5).display()

## Derive event date, event hour.

In [0]:
clickstream_df = clickstream_df.withColumn(
    "event_date", to_date("event_time")
).withColumn("event_hour", hour("event_time"))

display(clickstream_df.count())
clickstream_df.limit(5).display()

## Identify anonymous sessions

# Create Silver table

## Write to silver schema

In [0]:
clickstream_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.clickstream")

## Read silver.clickstream table

In [0]:
spark.read.table("ecommerce.silver.clickstream").display()